RQ2: What patterns of positive, neutral, and negative framing are used in German media when reporting on social movements, and how do these patterns vary across different news outlets? (Bild vs. Tagesschau)

Source: tagesschau_zdf_pbs_foxnews_bild_gkg_partitioned_full.csv 

Visualization: Grouped Bar Chart


In [4]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ─────────────────────────────────────────────────────────────────────
FOLDER   = r"C:\Users\User\Desktop\DataScienceProjectGroup8"
GKG_FILE = "tagesschau_zdf_pbs_foxnews_bild_gkg_partitioned_full.csv"

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ 1. LOAD DATA                                                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

df_raw = pd.read_csv(os.path.join(FOLDER, GKG_FILE),
                     on_bad_lines='skip', low_memory=False)

print(f"Total rows: {len(df_raw):,}")
print(f"Columns: {list(df_raw.columns)}")

url_col   = 'url'
tone_col  = 'V2Tone'
theme_col = None
date_col  = 'DATE'

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ 2. OUTLET LABELING                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def label_outlet(url):
    url = str(url).lower()
    if 'tagesschau.de' in url: return 'Tagesschau'
    elif 'bild.de' in url:     return 'Bild'
    else:                       return 'Other'

df_raw['Outlet'] = df_raw[url_col].apply(label_outlet)

print("\n── Records per outlet ──")
print(df_raw['Outlet'].value_counts().to_string())

df = df_raw[df_raw['Outlet'].isin(['Tagesschau', 'Bild'])].copy()
print(f"\nTagesschau + Bild records: {len(df):,}")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ 3. TONE PARSING                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def parse_tone(val):
    try:    return float(str(val).split(',')[0])
    except: return np.nan

def classify_tone(t):
    try:
        t = float(t)
        if t > 1.0:   return 'Positive'
        elif t < -1.0: return 'Negative'
        else:          return 'Neutral'
    except:
        return None

df['AvgTone']   = df[tone_col].apply(parse_tone)
df['ToneLabel'] = df['AvgTone'].apply(classify_tone)

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ 4. SOCIAL MOVEMENT FILTER (URL-Keywords only)                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

URL_KEYWORDS = [
    'protest', 'demonstration', 'demo-', 'streik', 'kundgebung',
    'bewegung', 'aktivist', 'fridays-for-future', 'klimaprotest',
    'verdi', 'gewerkschaft', 'arbeitskampf', 'blockade',
    'bauernprotest', 'klimastreik', 'buergerinitiative'
]

url_filter = df[url_col].fillna('').str.lower().str.contains(
    '|'.join(URL_KEYWORDS))

df_soc = df[url_filter].copy()

print(f"\n── Social movement articles found ──")
print(df_soc.groupby('Outlet').size().to_string())

df_soc_dedup = (df_soc.groupby([url_col, 'Outlet'])['AvgTone']
                .mean().reset_index())
df_soc_dedup['ToneLabel'] = df_soc_dedup['AvgTone'].apply(classify_tone)
df_soc_dedup = df_soc_dedup.dropna(subset=['ToneLabel'])

print(f"\n── After deduplication ──")
print(df_soc_dedup.groupby('Outlet').size().to_string())
print(f"\n── Mean AvgTone ──")
print(df_soc_dedup.groupby('Outlet')['AvgTone'].mean().round(3).to_string())

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ 5. AGGREGATE TONE DISTRIBUTION                                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

summary = (df_soc_dedup.groupby(['Outlet', 'ToneLabel'])
           .size().reset_index(name='Count'))
summary['Total'] = summary.groupby('Outlet')['Count'].transform('sum')
summary['Pct']   = summary['Count'] / summary['Total'] * 100

print("\n── Tone Distribution per Outlet (%) ──")
print(summary.to_string(index=False))

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ 6. GROUPED BAR CHART (interaktiv – Plotly)                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝

OUTLETS     = [o for o in ['Tagesschau', 'Bild'] if o in df_soc_dedup['Outlet'].values]
TONE_ORDER  = ['Negative', 'Neutral', 'Positive']
TS_COLOR    = '#1565C0'
BILD_COLOR  = '#D50000'

bar_colors = {
    'Tagesschau': {'Negative': '#1565C0', 'Neutral': '#1565C0', 'Positive': '#1565C0'},
    'Bild':       {'Negative': '#C62828', 'Neutral': '#C62828', 'Positive': '#C62828'},
}

fig = go.Figure()

for outlet in OUTLETS:
    pct_vals, count_vals = [], []
    for tone in TONE_ORDER:
        row = summary[(summary['Outlet'] == outlet) & (summary['ToneLabel'] == tone)]
        pct_vals.append(round(row['Pct'].values[0], 2) if len(row) else 0.0)
        count_vals.append(int(row['Count'].values[0]) if len(row) else 0)

    color     = TS_COLOR if outlet == 'Tagesschau' else BILD_COLOR
    bar_clrs  = [bar_colors[outlet][t] for t in TONE_ORDER]
    emoji     = '📺' if outlet == 'Tagesschau' else '🗞'

    fig.add_trace(go.Bar(
        name=outlet,
        x=TONE_ORDER,
        y=pct_vals,
        marker_color=bar_clrs,
        marker_line=dict(color='white', width=1.5),
        customdata=list(zip(count_vals, [f"{p:.1f}" for p in pct_vals])),
        hovertemplate=(
            f"<b>{emoji} {outlet} – %{{x}}</b><br>"
            "Anteil: %{customdata[1]}%<br>"
            "Artikel: %{customdata[0]}<extra></extra>"
        ),
        text=[f"{p:.1f}%" for p in pct_vals],
        textposition='outside',
        textfont=dict(size=12, color=color),
    ))

n_ts   = int(summary[summary['Outlet'] == 'Tagesschau']['Count'].sum()) \
         if 'Tagesschau' in OUTLETS else 0
n_bild = int(summary[summary['Outlet'] == 'Bild']['Count'].sum()) \
         if 'Bild' in OUTLETS else 0

fig.update_layout(
    barmode='group',
    bargap=0.25,
    bargroupgap=0.08,
    title=dict(
        text=(
            "Tone Distribution — "
            "<span style='color:#1565C0;font-weight:bold;'>Tagesschau</span>"
            " vs. "
            "<span style='color:#D50000;font-weight:bold;'>Bild</span>"
            " · Soziale Bewegungen"
        ),
        font=dict(size=18, family='Arial'),
        x=0.5, xanchor='center',
    ),
    xaxis=dict(title='Tone Category', tickfont=dict(size=13)),
    yaxis=dict(
        title='Share of Articles (%)',
        range=[0, 115],
        gridcolor='#f0f0f0',
        zeroline=False,
    ),
    legend=dict(orientation='h', y=1.12, x=0.5, xanchor='center', font=dict(size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=520,
    margin=dict(t=130, b=110, l=60, r=40),
    annotations=[dict(
        text=(
            f"Source: GDELT GKG | "
            f"Tagesschau n={n_ts}, Bild n={n_bild} | "
            "AvgTone > 1 = Positive, < −1 = Negative"
        ),
        xref='paper', yref='paper',
        x=0.5, y=-0.20,
        showarrow=False,
        font=dict(size=10, color='gray'),
        xanchor='center',
    )],
    hoverlabel=dict(bgcolor='white', font_size=13, bordercolor='#cccccc'),
)

fig.show()


Total rows: 292,678
Columns: ['DATE', 'SourceCommonName', 'url', 'V2Persons', 'V2Organizations', 'V2Tone']

── Records per outlet ──
Outlet
Other         158958
Bild          100502
Tagesschau     33218

Tagesschau + Bild records: 133,720

── Social movement articles found ──
Outlet
Bild          1533
Tagesschau    1035

── After deduplication ──
Outlet
Bild          1533
Tagesschau    1035

── Mean AvgTone ──
Outlet
Bild         -3.650
Tagesschau   -4.293

── Tone Distribution per Outlet (%) ──
    Outlet ToneLabel  Count  Total       Pct
      Bild  Negative   1253   1533 81.735160
      Bild   Neutral    169   1533 11.024136
      Bild  Positive    111   1533  7.240705
Tagesschau  Negative    929   1035 89.758454
Tagesschau   Neutral     91   1035  8.792271
Tagesschau  Positive     15   1035  1.449275


Interpretation of Social Movement Framing: Tagesschau vs. Bild

Dominant pattern — Negative framing in both outlets:
Both outlets frame social movements predominantly negatively. Tagesschau assigns a negative tone to 89.8% of its social movement articles (n=929), while Bild reaches 81.7% (n=1,253). This confirms that negative framing is the norm across both a public broadcaster and a tabloid outlet — neither outlet provides primarily neutral or positive coverage of social movements.

Key difference — Bild is comparatively less negative:
The most striking finding in the chart is the 8.1 percentage point gap in the negative bar — Tagesschau (89.8%) frames social movements more negatively than Bild (81.7%). This is counterintuitive, as one might expect a tabloid like Bild to be more sensationalist and negative. Instead, Bild shows noticeably more neutral (11.0% vs. 8.8%) and positive (7.2% vs. 1.4%) coverage than Tagesschau.

Positive framing — a clear outlet difference:
The positive bar reveals the sharpest contrast between the two outlets. Bild assigns a positive tone to 7.2% of social movement articles (n=111), which is five times higher than Tagesschau's 1.4% (n=15). This suggests that Bild occasionally frames social movements in a sympathetic or celebratory manner, while Tagesschau almost never does.

Summary:
The chart reveals that both German outlets apply a dominant negative framing to social movements, consistent with the protest paradigm in media studies. However, contrary to theoretical expectations, Tagesschau frames social movements more negatively than Bild, with significantly less neutral and positive coverage. Bild's higher share of positive and neutral articles suggests a more varied framing repertoire for social movement topics, possibly driven by populist alignment with certain protest movements.

In [18]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ZEITREIHE — komplett eigenständige Zelle, braucht nichts vorher       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ── Datei direkt einlesen ─────────────────────────────────────────────────────
FOLDER   = r"C:\Users\User\Desktop\DataScienceProjectGroup8"
GKG_FILE = "tagesschau_zdf_pbs_foxnews_bild_gkg_partitioned_full.csv"

_df = pd.read_csv(
    os.path.join(FOLDER, GKG_FILE),
    on_bad_lines='skip',
    low_memory=False,
    usecols=['DATE', 'url', 'V2Tone']   # nur nötige Spalten → schneller
)

print("Spalten:", _df.columns.tolist())
print(f"Zeilen:  {len(_df):,}")

# ── Outlet labeln ─────────────────────────────────────────────────────────────
def _label(url):
    url = str(url).lower()
    if 'tagesschau.de' in url: return 'Tagesschau'
    if 'bild.de'       in url: return 'Bild'
    return None

_df['Outlet'] = _df['url'].apply(_label)
_df = _df[_df['Outlet'].notna()].copy()
print(f"\nTagesschau: {(_df['Outlet']=='Tagesschau').sum():,}")
print(f"Bild:       {(_df['Outlet']=='Bild').sum():,}")

# ── Tone parsen ───────────────────────────────────────────────────────────────
_df['AvgTone'] = pd.to_numeric(
    _df['V2Tone'].astype(str).str.split(',').str[0],
    errors='coerce'
)

# ── Social Movement Filter ────────────────────────────────────────────────────
_KEYWORDS = (
    'protest|demonstration|demo-|streik|kundgebung|bewegung|aktivist|'
    'fridays-for-future|klimaprotest|verdi|gewerkschaft|arbeitskampf|'
    'blockade|bauernprotest|klimastreik|buergerinitiative'
)
_soc = _df[_df['url'].fillna('').str.lower().str.contains(_KEYWORDS)].copy()
print(f"\nSoziale Bewegungen — Tagesschau: {(_soc['Outlet']=='Tagesschau').sum()}")
print(f"Soziale Bewegungen — Bild:       {(_soc['Outlet']=='Bild').sum()}")

# ── Datum parsen (GDELT: YYYYMMDDHHMMSS) ──────────────────────────────────────
_soc['_date'] = pd.to_datetime(
    _soc['DATE'].astype(str).str[:8],
    format='%Y%m%d',
    errors='coerce'
)
_soc = _soc.dropna(subset=['_date', 'AvgTone'])
_soc['_month'] = _soc['_date'].dt.to_period('M')

# ── Monatliche Aggregation ────────────────────────────────────────────────────
_monthly = (
    _soc.groupby(['_month', 'Outlet'])['AvgTone']
    .agg(AvgTone='mean', Count='count')
    .reset_index()
)
_monthly['Date'] = _monthly['_month'].dt.to_timestamp()
_monthly = _monthly[_monthly['Count'] >= 3].sort_values('Date')

_ts   = _monthly[_monthly['Outlet'] == 'Tagesschau']
_bild = _monthly[_monthly['Outlet'] == 'Bild']

print(f"\nMonate Tagesschau: {len(_ts)} | Monate Bild: {len(_bild)}")

# ── Chart ─────────────────────────────────────────────────────────────────────
TS_COLOR, BILD_COLOR = '#1565C0', '#D50000'
fig = go.Figure()

if len(_ts) > 0:
    fig.add_trace(go.Scatter(
        x=_ts['Date'], y=_ts['AvgTone'],
        name='Tagesschau', mode='lines+markers',
        line=dict(color=TS_COLOR, width=3),
        marker=dict(size=7, color=TS_COLOR, line=dict(width=1.5, color='white')),
        fill='tozeroy', fillcolor='rgba(21,101,192,0.08)',
        customdata=np.stack([_ts['Count'], _ts['Date'].dt.strftime('%B %Y')], axis=1),
        hovertemplate=(
            "<b style='color:#1565C0'>📺 Tagesschau</b><br>"
            "%{customdata[1]}<br>Ø Tone: <b>%{y:.2f}</b><br>"
            "Artikel: %{customdata[0]}<extra></extra>"
        )
    ))
    fig.add_annotation(
        x=_ts['Date'].iloc[-1], y=_ts['AvgTone'].iloc[-1],
        text="<b>📺 Tagesschau</b>", showarrow=True, arrowhead=2,
        arrowcolor=TS_COLOR, font=dict(size=12, color=TS_COLOR, family='Arial Black'),
        bgcolor='white', bordercolor=TS_COLOR, borderwidth=1.5, ax=50, ay=30
    )

if len(_bild) > 0:
    fig.add_trace(go.Scatter(
        x=_bild['Date'], y=_bild['AvgTone'],
        name='Bild', mode='lines+markers',
        line=dict(color=BILD_COLOR, width=3, dash='dash'),
        marker=dict(size=8, color=BILD_COLOR, symbol='diamond',
                    line=dict(width=1.5, color='white')),
        fill='tozeroy', fillcolor='rgba(213,0,0,0.06)',
        customdata=np.stack([_bild['Count'], _bild['Date'].dt.strftime('%B %Y')], axis=1),
        hovertemplate=(
            "<b style='color:#D50000'>🗞 Bild</b><br>"
            "%{customdata[1]}<br>Ø Tone: <b>%{y:.2f}</b><br>"
            "Artikel: %{customdata[0]}<extra></extra>"
        )
    ))
    fig.add_annotation(
        x=_bild['Date'].iloc[-1], y=_bild['AvgTone'].iloc[-1],
        text="<b>🗞 Bild</b>", showarrow=True, arrowhead=2,
        arrowcolor=BILD_COLOR, font=dict(size=12, color=BILD_COLOR, family='Arial Black'),
        bgcolor='white', bordercolor=BILD_COLOR, borderwidth=1.5, ax=50, ay=-30
    )

fig.add_hline(y=0,  line_dash='dot',  line_color='gray',    opacity=0.5,
              annotation_text='Neutral (0)',
              annotation_position='bottom right',
              annotation_font=dict(size=11, color='gray'))
fig.add_hline(y=-1, line_dash='dash', line_color='#EF5350', opacity=0.4,
              annotation_text='Negativitätsschwelle (−1)',
              annotation_position='bottom right',
              annotation_font=dict(size=11, color='#EF5350'))

fig.update_layout(
    title=dict(
        text=(
            "Ø Tone über Zeit — "
            "<span style='color:#1565C0;font-weight:bold;'>Tagesschau</span>"
            " vs. "
            "<span style='color:#D50000;font-weight:bold;'>Bild</span>"
            " · Soziale Bewegungen"
            "</span>"
        ),
        font=dict(size=20, family='Arial'), x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='Monat', tickformat='%b %Y',
        rangeslider=dict(visible=True, thickness=0.08),
        rangeselector=dict(
            buttons=[
                dict(count=3,  label='3M',  step='month', stepmode='backward'),
                dict(count=6,  label='6M',  step='month', stepmode='backward'),
                dict(count=12, label='1J',  step='month', stepmode='backward'),
                dict(step='all', label='Alle'),
            ],
            bgcolor='white', activecolor=TS_COLOR, font=dict(size=12),
        ),
    ),
    yaxis=dict(title='Ø AvgTone Score', zeroline=False, gridcolor='#f0f0f0'),
    legend=dict(orientation='h', y=1.15, x=0.5, xanchor='center', font=dict(size=13)),
    height=540,
    margin=dict(t=140, b=80, l=60, r=130),
    plot_bgcolor='white', paper_bgcolor='white',
    hovermode='x unified',
    hoverlabel=dict(bgcolor='white', font_size=13, bordercolor='#cccccc'),
)

fig.show()


Spalten: ['DATE', 'url', 'V2Tone']
Zeilen:  292,678

Tagesschau: 33,218
Bild:       100,502

Soziale Bewegungen — Tagesschau: 1035
Soziale Bewegungen — Bild:       1533

Monate Tagesschau: 29 | Monate Bild: 29


Interpretation: Tone Over Time — Tagesschau vs. Bild

Persistent negativity across both outlets:
Both outlets remain consistently below the neutrality threshold throughout the entire observation period, confirming that negative framing of social movements is a stable, structural pattern — not an isolated event.

Tagesschau more negative than Bild:
Tagesschau's tone line runs systematically lower than Bild's across nearly all months (mean: −4.29 vs. −3.65). This mirrors the bar chart finding: the public broadcaster frames social movements more harshly than the tabloid on a sustained basis.

Bild reacts, Tagesschau stays consistent:
Bild shows higher month-to-month volatility, suggesting event-driven framing. Tagesschau's flatter trajectory points to a more institutionalized, uniform editorial stance toward social movements.